# Module 15: Feature Engineering — Solutions

Complete solutions to all exercises.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing, load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    OneHotEncoder, LabelEncoder, OrdinalEncoder,
    PolynomialFeatures, KBinsDiscretizer
)
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.feature_selection import (
    SelectKBest, f_classif, f_regression, RFE, VarianceThreshold
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, Ridge, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
print('Setup complete')

### Solution 1: Scaling Comparison

In [ ]:
housing = fetch_california_housing(as_frame=True)
X, y = housing.data[['MedInc', 'HouseAge']], housing.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scalers = {
    'None': None,
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler()
}

for name, scaler in scalers.items():
    if scaler is None:
        X_tr, X_te = X_train.values, X_test.values
    else:
        X_tr = scaler.fit_transform(X_train)
        X_te = scaler.transform(X_test)
    ridge = Ridge(alpha=1.0, random_state=42)
    ridge.fit(X_tr, y_train)
    score = ridge.score(X_te, y_test)
    print(f'{name}: R2 = {score:.4f}')

### Solution 2: Categorical Encoding

In [ ]:
titanic = sns.load_dataset('titanic').dropna(subset=['survived', 'embarked'])
X_cat = titanic[['embarked']].copy()
y_cat = titanic['survived']

X_train, X_test, y_train, y_test = train_test_split(X_cat, y_cat, test_size=0.2, random_state=42)

# OneHot
ohe = OneHotEncoder(sparse_output=False, drop='first')
X_train_ohe = ohe.fit_transform(X_train)
X_test_ohe = ohe.transform(X_test)
lr_ohe = LogisticRegression(max_iter=500).fit(X_train_ohe, y_train)
print(f'OneHot accuracy: {lr_ohe.score(X_test_ohe, y_test):.4f}')

# Label
le = LabelEncoder()
X_train_le = le.fit_transform(X_train.values.ravel())
X_test_le = le.transform(X_test.values.ravel())
lr_le = LogisticRegression(max_iter=500).fit(X_train_le.reshape(-1, 1), y_train)
print(f'Label accuracy: {lr_le.score(X_test_le.reshape(-1, 1), y_test):.4f}')

### Solution 3: Missing Value Imputation

In [ ]:
titanic = sns.load_dataset('titanic')
titanic = titanic[['survived', 'pclass', 'sex', 'age', 'fare']].copy()
titanic['sex'] = (titanic['sex'] == 'male').astype(int)

X = titanic.drop('survived', axis=1)
y = titanic['survived']

strategies = ['mean', 'median', 'most_frequent']
for strategy in strategies:
    imp = SimpleImputer(strategy=strategy)
    X_imp = imp.fit_transform(X)
    scores = cross_val_score(RandomForestClassifier(n_estimators=50, random_state=42),
                             X_imp, y, cv=5, scoring='accuracy')
    print(f'{strategy}: CV accuracy = {scores.mean():.4f} (+/- {scores.std():.4f})')

# KNN imputation
knn_imp = KNNImputer(n_neighbors=5)
X_knn = knn_imp.fit_transform(X)
scores = cross_val_score(RandomForestClassifier(n_estimators=50, random_state=42),
                         X_knn, y, cv=5, scoring='accuracy')
print(f'KNN (k=5): CV accuracy = {scores.mean():.4f} (+/- {scores.std():.4f})')

### Solution 4: Polynomial Features

In [ ]:
housing = fetch_california_housing(as_frame=True)
X = housing.data[['MedInc', 'HouseAge', 'AveRooms']]
y = housing.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Without polynomial
lr = LinearRegression()
lr.fit(X_train, y_train)
print(f'Without poly features: R2 = {lr.score(X_test, y_test):.4f}')

# With polynomial degree 2
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)
print(f'Polynomial features count: {X_train_poly.shape[1]}')

lr_poly = LinearRegression()
lr_poly.fit(X_train_poly, y_train)
print(f'With poly features: R2 = {lr_poly.score(X_test_poly, y_test):.4f}')

### Solution 5: Binning

In [ ]:
titanic = sns.load_dataset('titanic').dropna(subset=['survived', 'age'])
X_age = titanic[['age']].values
y_age = titanic['survived'].values

X_train, X_test, y_train, y_test = train_test_split(X_age, y_age, test_size=0.2, random_state=42)

# Raw age
lr = LogisticRegression(max_iter=500)
lr.fit(X_train, y_train)
print(f'Raw age accuracy: {lr.score(X_test, y_test):.4f}')

# Binned age
kbd = KBinsDiscretizer(n_bins=5, encode='onehot-dense', strategy='quantile')
X_train_binned = kbd.fit_transform(X_train)
X_test_binned = kbd.transform(X_test)
lr_binned = LogisticRegression(max_iter=500)
lr_binned.fit(X_train_binned, y_train)
print(f'Binned age accuracy: {lr_binned.score(X_test_binned, y_test):.4f}')

### Solution 6: Feature Selection

In [ ]:
iris = load_iris()
X, y, features = iris.data, iris.target, iris.feature_names

# SelectKBest
selector = SelectKBest(score_func=f_classif, k=2)
selector.fit(X, y)
skb_features = [features[i] for i in selector.get_support(indices=True)]
print(f'SelectKBest top 2: {skb_features}')

# RFE
rfe = RFE(estimator=LogisticRegression(max_iter=200, random_state=42), n_features_to_select=2)
rfe.fit(X, y)
rfe_features = [features[i] for i in rfe.get_support(indices=True)]
print(f'RFE top 2: {rfe_features}')

print(f'Methods agree: {set(skb_features) == set(rfe_features)}')

### Solution 7: Date Feature Engineering

In [ ]:
dates = pd.date_range('2023-01-01', periods=365, freq='D')
df = pd.DataFrame({'date': dates})
df['day_of_year'] = df['date'].dt.dayofyear
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['day_of_week'] = df['date'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
df['quarter'] = df['date'].dt.quarter

# Target: sine wave of day_of_year
df['target'] = np.sin(df['day_of_year'] * 2 * np.pi / 365) + np.random.normal(0, 0.1, 365)

X_date = df[['month', 'day', 'day_of_week', 'is_weekend', 'quarter']]
y_date = df['target']

X_train, X_test, y_train, y_test = train_test_split(X_date, y_date, test_size=0.2, random_state=42)
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
print(f'Date features Ridge R2: {ridge.score(X_test, y_test):.4f}')

### Solution 8: Text Feature Engineering

In [ ]:
reviews = [
    'amazing product works great',
    'terrible waste of money',
    'really love this item',
    'broken piece arrived defective',
    'excellent quality fast shipping',
    'poor quality not recommended',
    'best purchase ever',
    'disappointed with the product',
    'very satisfied with my purchase',
    'cheap material falls apart'
]
labels = [1, 0, 1, 0, 1, 0, 1, 0, 1, 0]

vectorizer = TfidfVectorizer(stop_words='english')
X_tfidf = vectorizer.fit_transform(reviews)

lr = LogisticRegression(max_iter=500)
lr.fit(X_tfidf, labels)

feature_names = vectorizer.get_feature_names_out()
coef_df = pd.DataFrame({'word': feature_names, 'coef': lr.coef_[0]})
print('Top 10 positive words (positive sentiment):')
print(coef_df.nlargest(10, 'coef').to_string(index=False))
print('\nTop 10 negative words (negative sentiment):')
print(coef_df.nsmallest(10, 'coef').to_string(index=False))

### Solution 9: Variance Threshold

In [ ]:
np.random.seed(42)
X_var = np.random.randn(100, 50)
X_var[:, :5] = 0.5
X_var[:, 5:10] = 0.0

selector = VarianceThreshold(threshold=0.01)
X_var_selected = selector.fit_transform(X_var)

removed = X_var.shape[1] - X_var_selected.shape[1]
print(f'Original features: {X_var.shape[1]}')
print(f'Features after VarianceThreshold: {X_var_selected.shape[1]}')
print(f'Features removed: {removed}')
print(f'Variances of first 10 features:\n{selector.variances_[:10].round(4)}')

### Solution 10: Full Feature Pipeline

In [ ]:
housing = fetch_california_housing(as_frame=True)
X, y = housing.data, housing.target

# Define transformers for different column groups
num_cols = ['MedInc', 'AveRooms']
age_col = ['HouseAge']
other_cols = [c for c in X.columns if c not in num_cols + age_col]

pipeline = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('scale_poly', Pipeline([
            ('scaler', StandardScaler()),
            ('poly', PolynomialFeatures(degree=2, include_bias=False))
        ]), num_cols),
        ('bin_age', KBinsDiscretizer(n_bins=5, encode='onehot-dense', strategy='quantile'), age_col),
        ('scale_other', StandardScaler(), other_cols)
    ])),
    ('select', SelectKBest(score_func=f_regression, k=5)),
    ('regressor', Ridge(alpha=1.0, random_state=42))
])

scores = cross_val_score(pipeline, X, y, cv=5, scoring='r2')
print('=== Full Feature Pipeline ===')
print(f'5-fold CV R2 scores: {np.round(scores, 4)}')
print(f'Mean R2: {scores.mean():.4f} (+/- {scores.std():.4f})')